In [ ]:
'''

'''

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [5]:
import random
from collections import Counter
words = open('names.txt', 'r').read().splitlines()

random.seed(12345)
random.shuffle(words)

top_8_names= [name for name, count in Counter(words).most_common(8)]
top_8_names

['jean', 'erin', 'ever', 'barrett', 'andrew', 'tai', 'milan', 'shelby']

In [7]:
import string
chars = ['.'] +list(string.ascii_lowercase) # ['.', a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, s, t, u, v, w, x, y, z]
string_to_index = {ch:i for i, ch in enumerate(chars)} # {'.': 0, 'a': 1, 'b': 2, ..., 'z': 26}
index_to_string = {i:ch for i, ch in enumerate(chars)} # {0: '.', 1: 'a', 2: 'b', ..., 26: 'z'}

In [31]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X = []  # input to the neural Network
Y = []  # labels for each example inside

for w in top_8_names:
  # print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = string_to_index[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(index_to_string[i] for i in context), '--->', index_to_string[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> j
..j ---> e
.je ---> a
jea ---> n
ean ---> .
... ---> e
..e ---> r
.er ---> i
eri ---> n
rin ---> .
... ---> e
..e ---> v
.ev ---> e
eve ---> r
ver ---> .
... ---> b
..b ---> a
.ba ---> r
bar ---> r
arr ---> e
rre ---> t
ret ---> t
ett ---> .
... ---> a
..a ---> n
.an ---> d
and ---> r
ndr ---> e
dre ---> w
rew ---> .
... ---> t
..t ---> a
.ta ---> i
tai ---> .
... ---> m
..m ---> i
.mi ---> l
mil ---> a
ila ---> n
lan ---> .
... ---> s
..s ---> h
.sh ---> e
she ---> l
hel ---> b
elb ---> y
lby ---> .


In [33]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([47, 3]), torch.int64, torch.Size([47]), torch.int64)

In [38]:
X

tensor([[ 0,  0,  0],
        [ 0,  0, 10],
        [ 0, 10,  5],
        [10,  5,  1],
        [ 5,  1, 14],
        [ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 18],
        [ 5, 18,  9],
        [18,  9, 14],
        [ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 22],
        [ 5, 22,  5],
        [22,  5, 18],
        [ 0,  0,  0],
        [ 0,  0,  2],
        [ 0,  2,  1],
        [ 2,  1, 18],
        [ 1, 18, 18],
        [18, 18,  5],
        [18,  5, 20],
        [ 5, 20, 20],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 14],
        [ 1, 14,  4],
        [14,  4, 18],
        [ 4, 18,  5],
        [18,  5, 23],
        [ 0,  0,  0],
        [ 0,  0, 20],
        [ 0, 20,  1],
        [20,  1,  9],
        [ 0,  0,  0],
        [ 0,  0, 13],
        [ 0, 13,  9],
        [13,  9, 12],
        [ 9, 12,  1],
        [12,  1, 14],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19,  8],
        [19,  8,  5],
        [ 8,  5, 12],
        [ 

In [ ]:
'''
part 1의 경우 문자를 One-hot Encoding 으로 표현하였음. 여기서는 학습가능한 벡터로 Embedding 한다.
신경망 학습의 대상이 되고, 문자의 특징을 숫자 벡터에 학습시킬 수 있다.
'''
C = torch.randn((27, 2)) # 27 문자로 제한, 2차원으로 간략하게. 

In [36]:
C[X].shape

torch.Size([47, 3, 2])

In [ ]:
X[12, 2]

tensor(22)

In [43]:
C[X][12,2]

tensor([-0.8254,  1.5999])

In [44]:
emb = C[X]
emb.shape

torch.Size([47, 3, 2])

In [46]:
W1 = torch.randn((6, 100)) # 6: 3 x 2. 바로 위 3개의 문자 받고, 각 문자는 2개의 값으로 만들었으니 6. 100은 100개의 neuron 이다.
b1 = torch.randn(100)

In [ ]:
# 연산을 위해 format 변경 필요. 아래에서 진행한다. emb.view(47, 6) 로 진행하는 과정을 보여줌.
emb @ W1 + b1

RuntimeError: mat1 and mat2 shapes cannot be multiplied (141x2 and 6x100)

In [ ]:
# 두 개 같은거임
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([47, 6])

In [52]:
emb.shape

torch.Size([47, 3, 2])

In [53]:
emb.view(47, 6) == torch.cat(torch.unbind(emb, 1), 1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [ ]:
'''
셋 다 같은거임. -1 을 넣으면 알아서 infer 해준다.
h: hidden layer를 만는다.
'''

# h = emb.view(47, 6) @ W1 + b1
# h = emb.view(emb.shape[0], 6) @ W1 + b1
h = emb.view(-1, 6) @ W1 + b1
h

tensor([[-0.3240,  0.7048, -0.7061,  ..., -3.2572, -0.2525, -3.9272],
        [-1.5702, -0.0222,  0.9023,  ..., -2.3054,  1.7409, -4.0296],
        [ 4.1229, -2.4579,  2.0327,  ..., -2.7468, -0.4707, -2.0243],
        ...,
        [ 0.8176,  3.1986, -1.4251,  ...,  1.8407,  0.9902, -0.0128],
        [ 3.4036, -2.9652, -1.0235,  ..., -1.2044,  1.7780, -0.3882],
        [-1.9763, -2.9372, -1.7103,  ..., -1.2660,  2.2088,  0.4685]])

In [59]:
h.shape

torch.Size([47, 100])

In [61]:
# final layer
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2
logits.shape

'torch.FloatTensor'

In [67]:
counts = logits.exp()
counts

tensor([[5.0670e-04, 2.7984e+07, 1.1131e+08,  ..., 1.9711e-03, 1.1035e-02,
         1.0300e+00],
        [4.7280e-13, 6.2943e+03, 2.4912e+00,  ..., 1.1912e+03, 5.5513e-03,
         4.3964e-08],
        [1.3451e-06, 1.3586e+19, 2.0305e-08,  ..., 1.8221e-16, 7.7744e+06,
         9.9321e+03],
        ...,
        [3.8638e-10, 7.9635e-11, 3.4118e+04,  ..., 9.8078e+00, 2.1419e-03,
         6.5074e-15],
        [3.5816e-13, 1.9280e+01, 1.6745e-11,  ..., 4.0469e-05, 6.9531e-06,
         1.2790e-08],
        [3.3128e-07, 1.2428e+01, 8.5104e-14,  ..., 6.5753e-08, 6.4963e-01,
         3.7985e+01]])

In [ ]:
prob = counts / counts.sum(1, keepdim=True)

In [80]:
loss = -prob[torch.arange(47), Y]
# loss = F.cross_entropy(logits, Y)
loss

tensor([-1.0912e-26, -5.5209e-20, -1.0000e+00, -4.6304e-06, -9.4254e-35,
        -2.6325e-36, -2.2851e-17, -1.4013e-45, -4.3617e-30, -0.0000e+00,
        -2.6325e-36, -5.0013e-29, -2.0959e-05, -3.6280e-24, -9.5311e-26,
        -3.1067e-16, -7.1448e-23, -1.9275e-12, -4.8417e-10, -4.9542e-06,
        -8.6502e-03, -1.2589e-03, -5.2919e-27, -7.8108e-17, -9.9943e-01,
        -7.0581e-30, -1.3805e-30, -9.2228e-26, -7.1168e-19, -5.0346e-32,
        -2.0812e-27, -3.3680e-18, -7.0065e-45, -3.7208e-28, -3.9826e-24,
        -1.3116e-42, -0.0000e+00, -0.0000e+00, -7.6633e-05, -1.0981e-28,
        -7.5969e-35, -3.2237e-21, -6.1846e-04, -7.9365e-26, -3.4911e-12,
        -8.4868e-21, -1.2321e-22])

In [ ]:
'''
loss = -prob[torch.arange(47), Y].log().mean() 와
F.cross_entropy 이 수학적으로 같은 의미이지만, cross_entropy를 사용하면 efficient
부동소수점으로 인해 연산 순서, 계산 방식에 따라 발생하는 문제들을 해결해줌
'''
# loss = -prob[torch.arange(47), Y].log().mean()
loss = F.cross_entropy(logits, Y)
loss

tensor(54.0136)

In [93]:
'''
not, rewrite above
'''
X.shape, Y.shape

(torch.Size([47, 3]), torch.Size([47]))

In [10]:
g = torch.Generator().manual_seed(12345)
C = torch.rand((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.rand((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [123]:
sum(p.nelement() for p in parameters)  # number of parameters in total.

3481

In [8]:
'''
8 개의 학습데이터가 아닌 words 단어 전체로 학습 시키기 위해 X, Y 다시 선언함.
'''

# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X = []  # input to the neural Network
Y = []  # labels for each example inside

for w in words:
  # print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = string_to_index[ch]
    X.append(context)
    Y.append(ix)
    # print(''.join(index_to_string[i] for i in context), '--->', index_to_string[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)
X.shape, Y.shape

(torch.Size([6072714, 3]), torch.Size([6072714]))

In [11]:
for p in parameters:
    p.requires_grad = True

In [ ]:
'''
아래 for loop range(100)를 돌리는데 각 loop 당 word size 가 6072714 이다보니,
4m 넘게 소요되고 노트북 열이 올라가는 것이 느껴짐.

(Full) Batch Gradient Descent
'''

for _ in range(10):
    # forward pass
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    # 주석부분은 cross_entropy로 대체
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdim=True)
    # loss = -prob[torch.arange(47), Y].log().mean()
    loss = F.cross_entropy(logits, Y)

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())

2.8006768226623535


In [ ]:
'''
일부 데이터로 mini-batch를 생성한다.

일부 데이터로 Gradient를 계산해도 될까?

기존 Full-batch에서의 Loss는 전체 데이터에 대한 평균.
일부 데이터를 균등하게 무작위 추출한다해도, 전체 Gradient와 정확하게 일치하지 않는다.
다만, 무작위 균등 추출한다면 기대값은 같다.
즉 일부를 균등 무작위 추출하여 구하면 노이즈가 있는 전체 Gradient의 추정값이라고 볼 수 있음.

반복적으로 새로운 데이터를 무작위 선택해 업데이트하면 전체 Loss를 감소시킨다.
정확한 Gradient 한 번을 구하는 데 많은 시간을 소비하는 것보다, 
충분히 유용한 근사 Gradient를 빠르게 계산하고 파라미터를 여러 번 업데이트하는 것이 학습 시간 측면에서 유리할 수 있다.

keyword: Stochastic Gradient Descent(SGD), Mini-batch SGD

위의 코드가 100 loop 로 4m 넘게 실행된 것에 비해 동일 100 loop 에서 0.0s에 실행이 완료되었다.
'''
for _ in range(1000):
    # mini batch 생성 - random 하게 32개 정수 생성
    ix = torch.randint(
        low=0,
        high=X.shape[0],
        size=(32,)
    )


    # forward pass
    # emb = C[X]  # 전체 단어 다 들어가있음 (6072714, 3, 2) -> (32, 3, 2)
    emb = C[X[ix]]

    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    # loss = F.cross_entropy(logits, Y): shape (6072714, 2) -> (32, 2)
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())

2.3872570991516113
